In [1]:
import json
from tqdm.notebook import tqdm
from ad_organoids.io import read_raw, window_signal, load_windows

# 01. MEA : Input/Ouput

This example walks through loading and windowing the MEA data. Since the .raw files are large, a utility function to window and save the signals into 1 seconds segments is provided. Once this is performed the slow read_raw function will not be required again allowing for memory efficient paralleziation for futre processing steps. Lastly, this notebook shows how to load the 1s windows back into memory as needed.

## Raw

Below, the raw signal and metadata are extract from the .raw file. This function is slow since there are > 1 billion raw int16's to read. Once we save out the signal into smaller windows, the data can be reloaded for future analysis more quickly and efficiently.

The signal array has the shape of:

```(nrows_well, ncols_well, nrows_electrodes, ncols_electrodes, ntimepoints)```

In [2]:
# Paths
dirpath = '/home/rphammonds/projects/ad_organoids'
raw_path = f'{dirpath}/data/plate4/plate4_midpoint_00/SL-TL-Plate4_96_4037-Baseline-20220406_000.raw'
carray_path = f'{dirpath}/channel_array.mat'

# Read in raw and meta data
sig, meta = read_raw(raw_path, channel_array_path=carray_path)

## Metadata

The meta object return from read_raw contains the sampling rate, scaling, units, and well labels for the .raw file.

In [3]:
# Convert array to list
meta['well_labels'] = meta['well_labels'].tolist()

# Save to json for later loading
with open(f'{dirpath}/data/plate4/meta.json', 'w') as fp:
    json.dump(meta, fp)

## Windowing Signal 

Below, the signal is windowed into 1 second intervals and saved for future access. The shape of the signal now becomes:

```(nwindows, nrows_well, ncols_well, nrows_electrodes, ncols_electrodes, ntimepoints)```

In [4]:
# Save the signal into 1s windows
fs = meta['fs'] * 1000

output_dir = f'{dirpath}/signals/plate4/plate4_midpoint_00'

window_signal(sig, fs, output_dir, win_len=1., compressed=True,
              n_jobs=-1, progress=tqdm)

  0%|          | 0/297 [00:00<?, ?it/s]

## Loading Windows

Once saved, the windows can be reloaded into memory for futher processing.

In [5]:
# Load the first 2 1s windows
sig = load_windows(output_dir, [0, 1])

In [6]:
# Rescale int16 data to Volts
scale = meta['scale']
sig = sig * scale